<a href="https://colab.research.google.com/github/vikramsingh456/ai-ml/blob/dev/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

df = pd.read_csv("/content/online_gaming_behavior_dataset.csv")

print("Dataset loaded successfully!")

print("\n First 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\n Column Names:")
print(df.columns)

print("\n Data Types:")
print(df.dtypes)

print("\n Dataset Information:")
print(df.info)

print("\n Missing Values:")
print(df.isnull().sum())

print("\nClass Distribution")
print(df["EngagementLevel"].value_counts())

if "PlayerID" in df.columns:
  df = df.drop("PlayerID", axis=1)

categorical_columns = df.select_dtypes(include=["object"]).columns

print("\nCategorical Columns:")
print(categorical_columns)


feature_categorical_columns = [
    col for col in categorical_columns
    if col != "EngagementLevel"
]

# Dictionary to store encoders
encoders = {}

for column in feature_categorical_columns:

  encoder = LabelEncoder()
  df[column] = encoder.fit_transform(df[column])
  encoders[column] = encoder

X = df.drop("EngagementLevel", axis=1)
y = df["EngagementLevel"]

print("\n Input Features (X):")
print(X.head())

print("\n Target Variable (y):")
print(y.head())

print("\nFeatures used by the model:")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n Training Data Shape:")
print(X_train.shape, y_train.shape)

print("\n Testing Data Shape:")
print(X_test.shape, y_test.shape)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print("\nRandom Forest Model Created:")

model.fit(X_train, y_train)

print("\n Model Training Completed:")

y_pred = model.predict(X_test)

print("\n First 10 Actual Values:")
print(y_test.head(10))

print("\n First 10 Predicted Values:")
print(y_pred[:10])



accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.2f}")

print("\n Accuracy Percentage:")
print(f"{accuracy * 100:.2f}%")

print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n Classification Report:")
print(classification_report(y_test, y_pred))


feature_importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importances.sort_values(by="Importance", ascending=False)
print("\n Feature Importance:")
print(feature_importance)

# Predict one existing gamer

one_gamer = X_test.iloc[[0]]

print("\nGamer Details:")
print(one_gamer)

prediction = model.predict(one_gamer)

print("\nPredicted Engagement Level:")
print(prediction[0])

#Compare with actual value

actual = y_test.iloc[0]

print("\nActual Engagement Level:")
print(actual)

probabilites = model.predict_proba(one_gamer)

print("\nClasses:")
print(model.classes_)

print("\nPrediction Probabilities:")
print(probabilites)

print("\nProbabilities for each class:")
for class_label, probability in zip(model.classes_, probabilites[0]):
  print(
      class_label,
      ":",
      round(probability * 100, 2),
      "%"
  )



















Dataset loaded successfully!

 First 5 rows:
   PlayerID  Age  Gender Location GameGenre  PlayTimeHours  InGamePurchases  \
0      9000   43    Male    Other  Strategy      16.271119                0   
1      9001   29  Female      USA  Strategy       5.525961                0   
2      9002   22  Female      USA    Sports       8.223755                0   
3      9003   35    Male      USA    Action       5.265351                1   
4      9004   33    Male   Europe    Action      15.531945                0   

  GameDifficulty  SessionsPerWeek  AvgSessionDurationMinutes  PlayerLevel  \
0         Medium                6                        108           79   
1         Medium                5                        144           11   
2           Easy               16                        142           35   
3           Easy                9                         85           57   
4         Medium                2                        131           95   

   AchievementsUn

In [7]:
# SMOTE Example

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from collections import Counter

# 1. Generate a highly imbalanced dummy dataset (99% Legitimate, 1% Fraud)
X, y = make_classification(
    n_samples=10000,
    n_features=5,
    weights=[0.99, 0.01],
    flip_y=0,
    random_state=42
)

print(f"Original dataset shape: {Counter(y)}")
# Output will show approx: {0: 9900, 1: 100} (0 = Legitimate, 1 = Fraud)

# 2. CRITICAL STEP: Split into Train and Test sets BEFORE oversampling
# This prevents data leakage into your evaluation set!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set before SMOTE: {Counter(y_train)}")


print("X shape")
print(X.shape)


print("Y Shape")
print(y.shape)
# 3. Apply SMOTE to the TRAINING data only
# This creates new synthetic examples instead of just repeating rows.
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Training set AFTER SMOTE: {Counter(y_train_resampled)}")
# Output will show perfectly balanced classes: {0: 6930, 1: 6930}

# 4. Train a Machine Learning Model on the balanced training data
model = RandomForestClassifier(random_state=42)
model.fit(X_train_resampled, y_train_resampled)

# 5. Evaluate the model using the untouched test data
y_pred = model.predict(X_test)

# Print a report focusing on Recall for Class 1 (Fraud)
print("\n--- Model Evaluation Report ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))


Original dataset shape: Counter({np.int64(0): 9900, np.int64(1): 100})
Training set before SMOTE: Counter({np.int64(0): 6930, np.int64(1): 70})
X shape
(10000, 5)
Y Shape
(10000,)
Training set AFTER SMOTE: Counter({np.int64(0): 6930, np.int64(1): 6930})

--- Model Evaluation Report ---
              precision    recall  f1-score   support

  Legitimate       1.00      0.98      0.99      2970
       Fraud       0.21      0.57      0.31        30

    accuracy                           0.97      3000
   macro avg       0.60      0.77      0.65      3000
weighted avg       0.99      0.97      0.98      3000

